In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import logging
import os
import sys

import h5py
import healpy as hp
import numpy as np

sys.path.append(os.path.join(os.getcwd(), ".."))
from scripts.utils import remove_mono_dipole, setup_logging, plot_predictions, trim_alms
from scripts import Core

from notebooks.ksw_joblib import KSW_joblib

from ksw import Cosmology, Shape
import camb
from scripts.estimator import icov_func, conv_beam_func, run_ksw_step

In [13]:
logger = setup_logging(__name__, level=logging.DEBUG)

## Elsner

In [14]:
core = Core(
    [
        "settings/elsner.json",
        "--nsims",
        "100",
        "--narray",
        "10",
        # "--polarizations",
        # "TE",
    ]
)

08-Aug-24 11:15:22 - scripts.core - DEBUG - Parsing CLI args: ['settings/elsner.json', '--nsims', '100', '--narray', '10']
08-Aug-24 11:15:22 - scripts.core - INFO - Loading settings from file settings/elsner.json
08-Aug-24 11:15:22 - scripts.core - DEBUG - Forcing setting 'nsims' to '100' due to CLI
08-Aug-24 11:15:22 - scripts.core - DEBUG - Forcing setting 'narray' to '10' due to CLI
08-Aug-24 11:15:22 - scripts.core - DEBUG - Found non-default value for 'cosmo_params': {'H0': 70.1, 'As': 2.457e-09, 'pivot_scalar': 0.002, 'ns': 0.96, 'ombh2': 0.02256, 'omch2': 0.1143, 'tau': 0.084} (default: {'As': 2.13e-09, 'ns': 0.9624, 'pivot_scalar': 0.05})
08-Aug-24 11:15:22 - scripts.core - DEBUG - Overriding cosmo param As from 2.13e-09 to 2.457e-09
08-Aug-24 11:15:22 - scripts.core - DEBUG - Overriding cosmo param ns from 0.9624 to 0.96
08-Aug-24 11:15:22 - scripts.core - DEBUG - Overriding cosmo param pivot_scalar from 0.05 to 0.002
08-Aug-24 11:15:22 - scripts.core - INFO - Running with se

08-Aug-24 11:15:22 - scripts.core - DEBUG - Slurm job name: jupyter-dev
08-Aug-24 11:15:22 - scripts.core - DEBUG - Number of available CPUs: 64
08-Aug-24 11:15:22 - scripts.core - DEBUG - Initializing cosmology
08-Aug-24 11:15:22 - scripts.core - DEBUG - Computing transfer functions
08-Aug-24 11:15:22 - scripts.core - DEBUG - Computing C_ell values
08-Aug-24 11:15:22 - scripts.core - DEBUG - Found non-default value for 'base_name': 'elsner' (default: 'l64_n512_ul-nn_Tx1000')
08-Aug-24 11:15:22 - scripts.core - DEBUG - Setting 'base_dir' not found, using default: 'data'
08-Aug-24 11:15:22 - scripts.core - DEBUG - Setting 'plot_dir' not found, using default: 'plots'
08-Aug-24 11:15:22 - scripts.core - DEBUG - Setting 'model_dir' not found, using default: 'models'
08-Aug-24 11:15:22 - scripts.core - DEBUG - Setting 'data_dir' not found, using default: 'data'
08-Aug-24 11:15:22 - scripts.core - DEBUG - Setting 'mc_dir' not found, using default: 'kswmc'
08-Aug-24 11:15:22 - scripts.core - 

In [15]:
cosmo_params = core.cosmo_params
cosmo = Cosmology(camb.set_params(**cosmo_params))
cosmo.compute_transfer(core.max_l)
cosmo.compute_c_ell()

noise = core.noise_ell[: core.npol]
beam = core.beam_ell[: core.npol]
c_ells = core.c_ells[: core.npol]

loc_shape = Shape.prim_local(cosmo_params["ns"], cosmo_params["pivot_scalar"])
cosmo.add_prim_reduced_bispectrum(loc_shape, core.radii)

ksw = KSW_joblib(
    cosmo.red_bispectra,
    icov_func(beam, noise, c_ells, core.npol),
    conv_beam_func(core),
    core.lmax,
    core.pols,
    core.precision,
)

print("num cpus available:", len(os.sched_getaffinity(0)))
thetas = int(np.floor(1.5 * core.lmax + 1)) // len(os.sched_getaffinity(0))

run_ksw_step(ksw, core, thetas)

08-Aug-24 11:15:30 - notebooks.ksw_joblib - INFO - Using KSW_joblib
num cpus available: 64
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Generating 100 ksw step alms
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Done
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Running KSW step, num steps: 100
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Sending alm step 0
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Sending alm step 1
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Sending alm step 2
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Sending alm step 3
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Sending alm step 4
08-Aug-24 11:15:30 - scripts.estimator-0 - DEBUG - Sending alm step 5
08-Aug-24 11:15:31 - scripts.estimator-0 - DEBUG - Sending alm step 6
08-Aug-24 11:15:31 - scripts.estimator-0 - DEBUG - Sending alm step 7
08-Aug-24 11:15:31 - scripts.estimator-0 - DEBUG - Sending alm step 8
08-Aug-24 11:15:31 - scripts.estimator-0 - DEBUG - Sending 

In [16]:
fisher = float(ksw.compute_fisher())
fisher, np.sqrt(1 / fisher)

(5.390497315294359e-05, 136.20265800886915)

In [17]:
num_estimates = 100

fnls = np.random.uniform(core.fnl_min, core.fnl_max + 1, num_estimates)
elsner_idxs = range(1, num_estimates)
elsner_fnls = fnls[elsner_idxs]


def alm_elsner_loader(idx):
    str_idx = str(idx).zfill(4)
    base1 = f"data/elsner/alm_l_{str_idx}_v3.fits"
    base2 = f"data/elsner/alm_nl_{str_idx}_v3.fits"

    alm_elsner_l = np.array(hp.read_alm(base1, hdu=1))
    alm_elsner_nl = np.array(hp.read_alm(base2, hdu=1))
    fnl = fnls[idx]

    alm = alm_elsner_l + fnl * alm_elsner_nl
    alm *= 2.7255 * 10 ** (6)  # convert elsner
    alm = remove_mono_dipole(alm)

    # need sto trim the values if we are using a lower lmax, throw error if asking for higher lmax
    lmax = hp.Alm.getlmax(alm.shape[-1])
    if lmax < core.lmax:
        # this could be done earlyer but this is just test code so not super important to optimize
        raise ValueError(
            "alm has lmax %s < %s which cannot be resolved", lmax, core.lmax
        )
    if lmax > core.lmax:
        # logger.debug("Trimming alm from lmax %s to %s", lmax, core.lmax)
        alm = trim_alms(alm, core.lmax)
    return alm


elsner_estimates = ksw.compute_estimate_batch(
    alm_elsner_loader, elsner_idxs, fisher=fisher, theta_batch=thetas
)

In [18]:
plot_predictions(elsner_fnls, elsner_estimates, fisher=fisher)

## Sim

In [19]:
alm_file = h5py.File(core.file_complete, "r", swmr=True, locking=False)
alms = alm_file["alm"]
fnls = alm_file["fnl"]


def estimator_loader(idx):
    """Loads in a single alm given an idx."""
    print("sending idx: %s, fnl: %s" % (idx, fnls[idx]))
    return alms[int(idx)]


elsner_idxs = range(core.total_sims)
sim_estimates = core.ksw.compute_estimate_batch(
    estimator_loader, elsner_idxs, fisher=fisher, theta_batch=thetas
)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'data/data/elsner.hdf5', errno = 2, error message = 'No such file or directory', flags = 40, o_flags = 0)

In [ ]:
plot_predictions(fnls[elsner_idxs], sim_estimates, fisher=fisher)

In [ ]:
from matplotlib import pyplot as plt

plt.scatter(elsner_fnls, elsner_estimates, label="elsner")
plt.scatter(fnls, sim_estimates, label="sim")

line = [min(fnls), max(fnls)]
plt.plot(line, line, color="red", linestyle="--", label="truth")
plt.xlabel("fnls")
plt.ylabel("Estimates")
plt.title("Scatter plot of Estimates vs fnls")
plt.show()